# 09 - Final Results Aggregation

This notebook combines all experimental results into final reports.

It collects:

- baseline evaluation results;
- merged adapter evaluation results;
- raw experiment metrics.

The notebook computes:

- mean values;
- standard deviations;
- aggregated summary tables.

In [ ]:
import exp_common as E
import pandas as pd
from pathlib import Path

base_raw = E.RESULTS_DIR/'baselines'/'baseline_metrics_raw.csv'
merged_raw = E.RESULTS_DIR/'merged'/'merged_metrics_raw.csv'
frames = []
if base_raw.exists():
    b = pd.read_csv(base_raw); b['group'] = 'baseline'; frames.append(b)
if merged_raw.exists():
    m = pd.read_csv(merged_raw); m['group'] = 'merged'; frames.append(m)
if not frames:
    raise FileNotFoundError('There are no result files')
all_df = pd.concat(frames, ignore_index=True)
all_df.to_csv(E.RESULTS_DIR/'all_metrics_raw.csv', index=False)
all_df.head()

In [ ]:
num_cols = [c for c in all_df.columns if c not in ['model', 'seed', 'group'] and pd.api.types.is_numeric_dtype(all_df[c])]
summary = all_df.groupby(['group','model'])[num_cols].agg(['mean','std'])
summary.to_csv(E.RESULTS_DIR/'all_metrics_mean_std.csv')
summary

In [ ]:
main_cols = ['logs_accuracy', 'logs_precision_anomaly', 'logs_recall_anomaly', 'logs_f1_anomaly', 'qa_semantic_similarity_mean', 'qa_llm_judge_mean_0_5']
rows = []
for (group, model), sub in all_df.groupby(['group','model']):
    row = {'group': group, 'model': model}
    for c in main_cols:
        if c in sub:
            row[c] = f"{sub[c].mean():.4f} ± {sub[c].std(ddof=0):.4f}"
    rows.append(row)
report = pd.DataFrame(rows).sort_values(['group','model'])
report.to_csv(E.RESULTS_DIR/'main_report_mean_std.csv', index=False)
md = report.to_markdown(index=False)
(E.RESULTS_DIR/'main_report_mean_std.md').write_text(md, encoding='utf-8')
report